In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Paths configuration
trans_csv_path = "Data/HI-Small_FANOUT_10M_test.csv"
output_accounts_path = "Data/HI-Small_FANOUT_10M_accounts.csv"

print(f"Reading transaction dataset from: {trans_csv_path}...")

# 2. Memory-efficient load of sender & receiver columns
usecols = ['From Bank', 'Account', 'To Bank', 'Account.1', 'Timestamp']
df = pd.read_csv(trans_csv_path, usecols=usecols)
print(f"Loaded {len(df):,} transactions.")

# 3. Extract sender & receiver accounts
df_senders = df[['From Bank', 'Account', 'Timestamp']].rename(
    columns={'From Bank': 'Bank ID', 'Account': 'Account Number'}
)
df_receivers = df[['To Bank', 'Account.1', 'Timestamp']].rename(
    columns={'To Bank': 'Bank ID', 'Account.1': 'Account Number'}
)

all_accounts = pd.concat([df_senders, df_receivers], ignore_index=True)
all_accounts['Bank ID'] = all_accounts['Bank ID'].astype(str).str.strip()
all_accounts['Account Number'] = all_accounts['Account Number'].astype(str).str.strip()

# 4. Compute earliest timestamp per account for deterministic Open Date derivation
earliest_ts = all_accounts.groupby('Account Number')['Timestamp'].min().to_dict()

# 5. Extract unique accounts
unique_accs = all_accounts[['Account Number', 'Bank ID']].drop_duplicates(subset=['Account Number']).reset_index(drop=True)
print(f"Extracted {len(unique_accs):,} unique accounts.")

# 6. Generate rich metadata attributes
cities = [
    "New York, USA", "London, UK", "Frankfurt, Germany", "Zurich, Switzerland",
    "Tokyo, Japan", "Singapore", "Toronto, Canada", "Sydney, Australia",
    "Paris, France", "Madrid, Spain", "Dubai, UAE", "Hong Kong"
]
entity_types = ["Corporation", "Sole Proprietorship", "Partnership", "Limited Liability Company", "Individual"]

bank_names, entity_ids, entity_names, account_types, kyc_statuses, acc_cities, open_dates = [], [], [], [], [], [], []

for _, row in unique_accs.iterrows():
    acc = row['Account Number']
    bank_id = row['Bank ID']
    h = abs(hash(acc))
    
    # Bank & Entity details
    bank_names.append(f"Global Bank #{bank_id}")
    e_type = entity_types[h % len(entity_types)]
    e_num = (h % 90000) + 10000
    entity_ids.append(f"ENT-{acc[:8]}")
    entity_names.append(f"{e_type} #{e_num}")
    
    # Account details
    account_types.append("Corporate Checking" if e_type != "Individual" else "Standard Checking")
    kyc_statuses.append("Verified KYC")
    acc_cities.append(cities[(h // 10) % len(cities)])
    
    # Open Date (1-3 years prior to account's first observed transfer)
    first_seen = earliest_ts.get(acc, "2022-09-09 21:15:00")
    try:
        dt = pd.to_datetime(first_seen)
        days_prior = 365 + (h % 730)
        open_dt = (dt - pd.Timedelta(days=days_prior)).strftime("%Y-%m-%d")
    except Exception:
        open_dt = "2020-04-12"
    open_dates.append(open_dt)

# 7. Assemble and export DataFrame
unique_accs['Bank Name'] = bank_names
unique_accs['Entity ID'] = entity_ids
unique_accs['Entity Name'] = entity_names
unique_accs['Account Type'] = account_types
unique_accs['KYC Status'] = kyc_statuses
unique_accs['City'] = acc_cities
unique_accs['Open Date'] = open_dates

ordered_cols = [
    'Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name',
    'Account Type', 'KYC Status', 'City', 'Open Date'
]
unique_accs = unique_accs[ordered_cols]

os.makedirs(os.path.dirname(output_accounts_path) or ".", exist_ok=True)
unique_accs.to_csv(output_accounts_path, index=False)
print(f"✅ Successfully created accounts CSV ({len(unique_accs):,} accounts) at: {output_accounts_path}")
